In [58]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
import random
import math
import numpy as np
import matplotlib.pyplot as plt




# Problem representation

We start off by defining the data structures we will use to represent the problem. 

## Customer

In [59]:
# Customer is defined by a location, x and y coordinates and a demand

@dataclass(frozen=True)
class Customer:
    cid: int
    x: float
    y: float
    demand: float



## Problem instance

In [60]:
# Instance is defined by a unique id, (optionally) a best-known solution, n customers and p facilities, which each has
# a capacity, a list of all customers and a distance matrix dist. 

@dataclass
class Instance:
    iid: int
    best_known: Optional[float]
    n: int
    p: int
    capacity: float
    customers: List[Customer]
    dist: np.ndarray  



## Individual/Genome

In [61]:
# Individual consist of a list of medians, representing the "genome" of the solution; an array of which customer is 
# assigned to which facility, objective, a tuple of how well the solution fits both constraints, a rank in the population's 
# pareto dominance and crowding  
@dataclass
class Individual:
    medians: List[int]                 
    objectives: Tuple[float, float]    
    rank: int = 10**9
    crowding: float = 0.0


## Utility functions 

These functions are for parsing all instances from the .txt files of benchmark instances. 

In [62]:
def parse_instances_from_text(text):

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    i = 0
    instances: List[Instance] = []

    while i < len(lines):
        a = lines[i].split()
        if len(a) < 2:
            raise ValueError(f"Bad header line: {lines[i]}")
        iid = int(a[0])
        best_known = float(a[1])
        i += 1

        b = lines[i].split()
        if len(b) < 3:
            raise ValueError(f"Bad size line: {lines[i]}")
        n = int(b[0])
        p = int(b[1])
        cap = float(b[2])
        i += 1

        customers: List[Customer] = []
        for _ in range(n):
            c = lines[i].split()
            if len(c) < 4:
                raise ValueError(f"Bad customer line: {lines[i]}")
            cid = int(c[0])
            x = float(c[1])
            y = float(c[2])
            d = float(c[3])
            customers.append(Customer(cid=cid, x=x, y=y, demand=d))
            i += 1

        dist = compute_distance_matrix(customers, metric="euclidean", rounding="floor")
        instances.append(Instance(iid=iid, best_known=best_known, n=n, p=p, capacity=cap,
                                  customers=customers, dist=dist))
    return instances


def parse_instances_from_file(path: str) -> List[Instance]:
    with open(path, "r", encoding="utf-8") as f:
        return parse_instances_from_text(f.read())


#distance /config
def compute_distance_matrix(customers, metric="euclidean", rounding="round"):
    n = len(customers)
    coords = np.array([(c.x, c.y) for c in customers], dtype=float)
    dx = coords[:, None, 0] - coords[None, :, 0]
    dy = coords[:, None, 1] - coords[None, :, 1]

    if metric == "euclidean":
        dist = np.sqrt(dx * dx + dy * dy)
    elif metric == "manhattan":
        dist = np.abs(dx) + np.abs(dy)
    else:
        raise ValueError("metric must be 'euclidean' or 'manhattan'")

    if rounding == "none":
        return dist
    if rounding == "floor":
        return np.floor(dist)
    if rounding == "ceil":
        return np.ceil(dist)
    if rounding == "round":
        return np.floor(dist + 0.5)  # standard .5 up

    raise ValueError("rounding must be none/floor/ceil/round")



In [64]:
def assignment_cost(dist_ij: float, demand: float, demand_weighted: bool = True) -> float:
    return dist_ij * demand if demand_weighted else dist_ij

# Calculates the objectives of a given individual
def decode_and_evaluate(
    inst: Instance,
    medians: List[int],
    demand_weighted: bool = True,
    customer_order: str = "desc_demand"  # "desc_demand" | "id"
):

    n, p, cap = inst.n, inst.p, inst.capacity
    if len(medians) != p or len(set(medians)) != p:
        # invalid genotype -> huge penalty
        return (1e18, 1e18)

    loads = {m: 0.0 for m in medians}
    cost = 0.0

    # choose assignment order (helps feasibility)
    if customer_order == "desc_demand":
        order = sorted(range(n), key=lambda i: inst.customers[i].demand, reverse=True)
    else:
        order = list(range(n))

    for ci in order:
        dmd = inst.customers[ci].demand

        # try nearest feasible median first
        # (p is small, simple scan is fine)
        best_feasible = None
        best_feasible_dist = float("inf")
        best_any = None
        best_any_dist = float("inf")

        for m in medians:
            dij = float(inst.dist[ci, m])

            if dij < best_any_dist:
                best_any_dist = dij
                best_any = m

            if loads[m] + dmd <= cap and dij < best_feasible_dist:
                best_feasible_dist = dij
                best_feasible = m

        chosen = best_feasible if best_feasible is not None else best_any
        loads[chosen] += dmd
        cost += assignment_cost(float(inst.dist[ci, chosen]), dmd, demand_weighted)

    violation = 0.0
    for m in medians:
        violation += max(0.0, loads[m] - cap)

    return (cost, violation)


# NSGA-II core, dominance, sorting, crowding
def constraint_dominates(a, b):

    cost_a, viol_a = a.objectives
    cost_b, viol_b = b.objectives

    feasible_a = (viol_a <= 1e-12)
    feasible_b = (viol_b <= 1e-12)

    if feasible_a and not feasible_b:
        return True
    if feasible_b and not feasible_a:
        return False
    if not feasible_a and not feasible_b:
        return viol_a < viol_b

    # both feasible: minimize cost (and violation ties)
    return (cost_a <= cost_b and viol_a <= viol_b) and (cost_a < cost_b or viol_a < viol_b)


def fast_non_dominated_sort(pop):
    S: Dict[int, List[int]] = {}
    n_dom = [0] * len(pop)
    fronts: List[List[int]] = []

    for i in range(len(pop)):
        S[i] = []
        n_dom[i] = 0
        for j in range(len(pop)):
            if i == j:
                continue
            if constraint_dominates(pop[i], pop[j]):
                S[i].append(j)
            elif constraint_dominates(pop[j], pop[i]):
                n_dom[i] += 1

        if n_dom[i] == 0:
            pop[i].rank = 0

    current = [i for i in range(len(pop)) if n_dom[i] == 0]
    fronts.append(current)

    r = 0
    while fronts[r]:
        next_front: List[int] = []
        for i in fronts[r]:
            for j in S[i]:
                n_dom[j] -= 1
                if n_dom[j] == 0:
                    pop[j].rank = r + 1
                    next_front.append(j)
        r += 1
        fronts.append(next_front)

    fronts.pop()  # last empty
    return [[pop[i] for i in f] for f in fronts]


def crowding_distance(front):
    if not front:
        return
    m = 2  # number of objectives: (cost, violation)
    for ind in front:
        ind.crowding = 0.0

    for k in range(m):
        front.sort(key=lambda ind: ind.objectives[k])
        front[0].crowding = float("inf")
        front[-1].crowding = float("inf")
        minv = front[0].objectives[k]
        maxv = front[-1].objectives[k]
        if abs(maxv - minv) < 1e-18:
            continue
        for i in range(1, len(front) - 1):
            prevv = front[i - 1].objectives[k]
            nextv = front[i + 1].objectives[k]
            front[i].crowding += (nextv - prevv) / (maxv - minv)


def binary_tournament(pop):
    a, b = random.sample(pop, 2)
    if a.rank < b.rank:
        return a
    if b.rank < a.rank:
        return b
    # same rank: higher crowding wins
    return a if a.crowding > b.crowding else b


def random_individual(inst: Instance) -> Individual:
    medians = random.sample(range(inst.n), inst.p)
    return Individual(medians=medians, objectives=(1e18, 1e18))


def crossover_set(parent1, parent2, inst):

    p = inst.p
    s1 = parent1.medians[:]
    s2 = parent2.medians[:]
    k = random.randint(1, p - 1)

    child1 = random.sample(s1, k) + random.sample(s2, p - k)
    child2 = random.sample(s2, k) + random.sample(s1, p - k)

    child1 = repair_unique(child1, inst.n, p)
    child2 = repair_unique(child2, inst.n, p)
    return child1, child2


def mutate_swap(medians, inst, pm):
    if random.random() > pm:
        return medians
    p = inst.p
    chosen = medians[:]
    idx = random.randrange(p)
    current = set(chosen)
    candidates = [i for i in range(inst.n) if i not in current]
    if not candidates:
        return chosen
    chosen[idx] = random.choice(candidates)
    return repair_unique(chosen, inst.n, p)


def repair_unique(medians, n, p):

    med = medians[:p]
    seen = set()
    cleaned: List[int] = []
    for x in med:
        if x not in seen and 0 <= x < n:
            cleaned.append(x)
            seen.add(x)
    while len(cleaned) < p:
        cand = random.randrange(n)
        if cand not in seen:
            cleaned.append(cand)
            seen.add(cand)
    return cleaned



# NSGA2 Algorithm 



In [65]:
class NSGA2Solver:
    def __init__(
        self,
        pop_size: int = 200,
        generations: int = 400,
        pc: float = 0.9,
        pm: float = 0.2,
        demand_weighted: bool = True,
        customer_order: str = "desc_demand",
        seed: int = 1234
    ):
        self.pop_size = pop_size
        self.generations = generations
        self.pc = pc
        self.pm = pm
        self.demand_weighted = demand_weighted
        self.customer_order = customer_order
        self.seed = seed

    def evaluate(self, inst, ind):
        ind.objectives = decode_and_evaluate(
            inst,
            ind.medians,
            demand_weighted=self.demand_weighted,
            customer_order=self.customer_order,
        )

    def run(self, inst):
        random.seed(self.seed)
        np.random.seed(self.seed)

        # init
        pop = [random_individual(inst) for _ in range(self.pop_size)]
        for ind in pop:
            self.evaluate(inst, ind)

        def best_feasible_cost(population):
            feas = [ind for ind in population if ind.objectives[1] <= 1e-12]
            if not feas:
                return None
            return min(feas, key=lambda ind: ind.objectives[0]).objectives[0]

        history = []
        history.append(best_feasible_cost(pop))   # gen 0

        # rank/crowding init
        fronts = fast_non_dominated_sort(pop)
        for f in fronts:
            crowding_distance(f)

        for gen in range(self.generations):
            # offspring
            offspring: List[Individual] = []
            while len(offspring) < self.pop_size:
                p1 = binary_tournament(pop)
                p2 = binary_tournament(pop)

                if random.random() < self.pc:
                    c1_med, c2_med = crossover_set(p1, p2, inst)
                else:
                    c1_med, c2_med = p1.medians[:], p2.medians[:]

                c1_med = mutate_swap(c1_med, inst, self.pm)
                c2_med = mutate_swap(c2_med, inst, self.pm)

                c1 = Individual(medians=c1_med, objectives=(1e18, 1e18))
                c2 = Individual(medians=c2_med, objectives=(1e18, 1e18))
                self.evaluate(inst, c1)
                self.evaluate(inst, c2)
                offspring.append(c1)
                if len(offspring) < self.pop_size:
                    offspring.append(c2)

            # combine + select next gen
            combined = pop + offspring
            fronts = fast_non_dominated_sort(combined)

            new_pop: List[Individual] = []
            for f in fronts:
                crowding_distance(f)
                if len(new_pop) + len(f) <= self.pop_size:
                    new_pop.extend(f)
                else:
                    # fill remainder by crowding
                    f.sort(key=lambda ind: ind.crowding, reverse=True)
                    need = self.pop_size - len(new_pop)
                    new_pop.extend(f[:need])
                    break

            pop = new_pop

            history.append(best_feasible_cost(pop))  # after this generation

        # final nondominated front
        fronts = fast_non_dominated_sort(pop)
        pareto_front = fronts[0]
        crowding_distance(pareto_front)

        best_feasible = min(
            (ind for ind in pop if ind.objectives[1] <= 1e-12),
            key=lambda ind: ind.objectives[0],
            default=min(pop, key=lambda ind: (ind.objectives[1], ind.objectives[0]))
        )
        return pareto_front, best_feasible, history



In [66]:


def run_nsga2(path: str, instance_id: Optional[int] = None, **solver_kwargs):
    instances = parse_instances_from_file(path)
    if instance_id is not None:
        instances = [inst for inst in instances if inst.iid == instance_id]
        if not instances:
            raise ValueError(f"No instance with id={instance_id} in file.")

    solver = NSGA2Solver(**solver_kwargs)

    histories = {}

    for inst in instances:
        front, best, history = solver.run(inst)
        cost, viol = best.objectives

        print(f"\nInstance {inst.iid} (best-known: {inst.best_known})")
        print(f"  n={inst.n}, p={inst.p}, cap={inst.capacity}")
        print(f"  Best found: cost={cost:.4f}, violation={viol:.4f}")
        print(f"  Medians (0-based indices): {best.medians}")
        print(f"  Pareto front size: {len(front)}")

        front_sorted = sorted(front, key=lambda ind: (ind.objectives[1], ind.objectives[0]))
        print("  Front sample (cost, violation):")
        for ind in front_sorted[:min(10, len(front_sorted))]:
            print(f"    ({ind.objectives[0]:.4f}, {ind.objectives[1]:.4f})")

        histories[inst.iid] = history

    if instance_id is not None:
        return histories[instance_id]
    return histories




In [52]:
#SPEA2

In [67]:
# Individual class requires slightly different fields for SPEA than NGSA; changed crowding and rank to fitness
@dataclass
class Individual:
    medians: List[int]
    objectives: Tuple[float, float]
    fitness: float = float("inf")           


# Parsing
def parse_instances_from_text(text: str, metric="euclidean", rounding="floor") -> List[Instance]:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    i = 0
    instances: List[Instance] = []

    while i < len(lines):
        a = lines[i].split()
        iid = int(a[0])
        best_known = float(a[1])
        i += 1

        b = lines[i].split()
        n = int(b[0])
        p = int(b[1])
        cap = float(b[2])
        i += 1

        customers: List[Customer] = []
        for _ in range(n):
            c = lines[i].split()
            customers.append(Customer(int(c[0]), float(c[1]), float(c[2]), float(c[3])))
            i += 1

        dist = compute_distance_matrix(customers, metric=metric, rounding=rounding)
        instances.append(Instance(iid, best_known, n, p, cap, customers, dist))

    return instances


def parse_instances_from_file(path: str, metric="euclidean", rounding="floor") -> List[Instance]:
    with open(path, "r", encoding="utf-8") as f:
        return parse_instances_from_text(f.read(), metric=metric, rounding=rounding)


# Distances
def compute_distance_matrix(customers: List[Customer], metric="euclidean", rounding="floor") -> np.ndarray:
    coords = np.array([(c.x, c.y) for c in customers], dtype=float)
    dx = coords[:, None, 0] - coords[None, :, 0]
    dy = coords[:, None, 1] - coords[None, :, 1]

    if metric == "euclidean":
        dist = np.sqrt(dx * dx + dy * dy)
    elif metric == "manhattan":
        dist = np.abs(dx) + np.abs(dy)
    else:
        raise ValueError("metric must be 'euclidean' or 'manhattan'")

    if rounding == "none":
        return dist
    if rounding == "floor":
        return np.floor(dist)
    if rounding == "ceil":
        return np.ceil(dist)
    if rounding == "round":
        return np.floor(dist + 0.5)
    raise ValueError("rounding must be none/floor/ceil/round")



# Decode / evaluate

def assignment_cost(dist_ij: float, demand: float, demand_weighted: bool) -> float:
    return dist_ij * demand if demand_weighted else dist_ij


def decode_and_evaluate(
    inst: Instance,
    medians: List[int],
    demand_weighted: bool = False,
    customer_order: str = "desc_demand",
) -> Tuple[float, float]:
    n, p, cap = inst.n, inst.p, inst.capacity
    if len(medians) != p or len(set(medians)) != p:
        return (1e18, 1e18)

    loads = {m: 0.0 for m in medians}
    cost = 0.0

    if customer_order == "desc_demand":
        order = sorted(range(n), key=lambda i: inst.customers[i].demand, reverse=True)
    else:
        order = list(range(n))

    for ci in order:
        demandd = inst.customers[ci].demand

        best_feas = None
        best_feas_d = float("inf")
        best_any = None
        best_any_d = float("inf")

        for m in medians:
            dij = float(inst.dist[ci, m])

            if dij < best_any_d:
                best_any_d = dij
                best_any = m

            if loads[m] + dmd <= cap and dij < best_feas_d:
                best_feas_d = dij
                best_feas = m

        chosen = best_feas if best_feas is not None else best_any
        loads[chosen] += dmd
        cost += assignment_cost(float(inst.dist[ci, chosen]), dmd, demand_weighted)

    violation = sum(max(0.0, loads[m] - cap) for m in medians)
    return (cost, violation)



def constraint_dominates(a: Individual, b: Individual) -> bool:
    ca, va = a.objectives
    cb, vb = b.objectives

    fa = (va <= 1e-12)
    fb = (vb <= 1e-12)

    if fa and not fb:
        return True
    if fb and not fa:
        return False
    if not fa and not fb:
        return va < vb

    return (ca <= cb and va <= vb) and (ca < cb or va < vb)


def repair_unique(medians: List[int], n: int, p: int) -> List[int]:
    med = medians[:p]
    seen = set()
    out: List[int] = []
    for x in med:
        if 0 <= x < n and x not in seen:
            out.append(x)
            seen.add(x)
    while len(out) < p:
        cand = random.randrange(n)
        if cand not in seen:
            out.append(cand)
            seen.add(cand)
    return out


def random_individual(inst: Instance) -> Individual:
    medians = random.sample(range(inst.n), inst.p)
    return Individual(medians=medians, objectives=(1e18, 1e18))


def crossover_set(p1: Individual, p2: Individual, inst: Instance) -> Tuple[List[int], List[int]]:
    p = inst.p
    k = random.randint(1, p - 1)
    c1 = random.sample(p1.medians, k) + random.sample(p2.medians, p - k)
    c2 = random.sample(p2.medians, k) + random.sample(p1.medians, p - k)
    return repair_unique(c1, inst.n, p), repair_unique(c2, inst.n, p)


def mutate_swap(medians: List[int], inst: Instance, pm: float) -> List[int]:
    if random.random() > pm:
        return medians
    p = inst.p
    out = medians[:]
    idx = random.randrange(p)
    current = set(out)
    candidates = [i for i in range(inst.n) if i not in current]
    if candidates:
        out[idx] = random.choice(candidates)
    return repair_unique(out, inst.n, p)


def objective_distance(a: Individual, b: Individual) -> float:
    # Euclidean distance in objectiwe space
    (c1, v1) = a.objectives
    (c2, v2) = b.objectives
    return math.sqrt((c1 - c2) ** 2 + (v1 - v2) ** 2)


def spea2_fitness(pop: List[Individual]) -> None:

    N = len(pop)
    dom = [[False] * N for _ in range(N)]
    strength = [0] * N

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            if constraint_dominates(pop[i], pop[j]):
                dom[i][j] = True
                strength[i] += 1

    raw = [0] * N
    for i in range(N):
        s = 0
        for j in range(N):
            if j != i and dom[j][i]:
                s += strength[j]
        raw[i] = s

    # Density
    k = int(math.sqrt(N))
    k = max(1, k)
    # pairwise distances
    dmat = [[0.0] * N for _ in range(N)]
    for i in range(N):
        for j in range(i + 1, N):
            d = objective_distance(pop[i], pop[j])
            dmat[i][j] = d
            dmat[j][i] = d

    density = [0.0] * N
    for i in range(N):
        ds = sorted(dmat[i][j] for j in range(N) if j != i)
        sigma_k = ds[min(k - 1, len(ds) - 1)] if ds else 0.0
        density[i] = 1.0 / (sigma_k + 2.0)

    for i in range(N):
        pop[i].fitness = raw[i] + density[i]


def spea2_truncate(archive: List[Individual], target_size: int) -> List[Individual]:
    A = archive[:]
    while len(A) > target_size:
        m = len(A)
        # distance lists
        dist_lists: List[List[float]] = []
        for i in range(m):
            ds = [objective_distance(A[i], A[j]) for j in range(m) if j != i]
            ds.sort()
            dist_lists.append(ds)

        # find index to remove: smallest lexicographic distance list
        remove_idx = 0
        for i in range(1, m):
            if dist_lists[i] < dist_lists[remove_idx]:
                remove_idx = i

        A.pop(remove_idx)

    return A


class SPEA2Solver:
    def __init__(
        self,
        pop_size: int = 200,
        archive_size: int = 200,
        generations: int = 400,
        pc: float = 0.9,
        pm: float = 0.25,
        demand_weighted: bool = False,
        customer_order: str = "desc_demand",
        seed: int = 1234,
    ):
        self.pop_size = pop_size
        self.archive_size = archive_size
        self.generations = generations
        self.pc = pc
        self.pm = pm
        self.demand_weighted = demand_weighted
        self.customer_order = customer_order
        self.seed = seed

    def evaluate(self, inst: Instance, ind: Individual) -> None:
        ind.objectives = decode_and_evaluate(
            inst,
            ind.medians,
            demand_weighted=self.demand_weighted,
            customer_order=self.customer_order,
        )

    def environmental_selection(self, union: List[Individual]) -> List[Individual]:
        spea2_fitness(union)

        # 1) take all with fitness < 1 into archive
        archive = [ind for ind in union if ind.fitness < 1.0]

        # 2) if too many, truncate
        if len(archive) > self.archive_size:
            archive = spea2_truncate(archive, self.archive_size)

        # 3) if too few, fill with best fitness individuals
        if len(archive) < self.archive_size:
            rest = [ind for ind in union if ind not in archive]
            rest.sort(key=lambda ind: ind.fitness)
            archive.extend(rest[: self.archive_size - len(archive)])

        return archive

    def tournament(self, archive: List[Individual]) -> Individual:
        a, b = random.sample(archive, 2)
        return a if a.fitness < b.fitness else b
    
    

    def run(self, inst):
        random.seed(self.seed)
        np.random.seed(self.seed)

        pop = [random_individual(inst) for _ in range(self.pop_size)]
        for ind in pop:
            self.evaluate(inst, ind)

        archive = []
        history = []

        def best_feasible_cost(arch):
            feas = [ind for ind in arch if ind.objectives[1] <= 1e-12]
            if feas:
                return min(feas, key=lambda ind: ind.objectives[0]).objectives[0]
            return None

        # record gen 0 (no archive yet -> use pop)
        feas0 = [ind for ind in pop if ind.objectives[1] <= 1e-12]
        history.append(min(feas0, key=lambda ind: ind.objectives[0]).objectives[0] if feas0 else None)

        for _ in range(self.generations):
            union = pop + archive
            archive = self.environmental_selection(union)

            # record after selection
            history.append(best_feasible_cost(archive))

            offspring = []
            while len(offspring) < self.pop_size:
                p1 = self.tournament(archive)
                p2 = self.tournament(archive)

                if random.random() < self.pc:
                    c1_med, c2_med = crossover_set(p1, p2, inst)
                else:
                    c1_med, c2_med = p1.medians[:], p2.medians[:]

                c1_med = mutate_swap(c1_med, inst, self.pm)
                c2_med = mutate_swap(c2_med, inst, self.pm)

                c1 = Individual(c1_med, (1e18, 1e18))
                c2 = Individual(c2_med, (1e18, 1e18))
                self.evaluate(inst, c1)
                self.evaluate(inst, c2)
                offspring.append(c1)
                if len(offspring) < self.pop_size:
                    offspring.append(c2)

            pop = offspring

        best = min(
            (ind for ind in archive if ind.objectives[1] <= 1e-12),
            key=lambda ind: ind.objectives[0],
            default=min(archive, key=lambda ind: (ind.objectives[1], ind.objectives[0]))
        )

        archive_sorted = sorted(archive, key=lambda ind: (ind.objectives[1], ind.objectives[0], ind.fitness))
        return archive_sorted, best, history



def run_spea2(path, instance_id, metric, rounding, **solver_kwargs):
    instances = parse_instances_from_file(path, metric=metric, rounding=rounding)
    if instance_id is not None:
        instances = [inst for inst in instances if inst.iid == instance_id]
        if not instances:
            raise ValueError(f"No instance with id={instance_id} in file.")

    solver = SPEA2Solver(**solver_kwargs)

    histories = {}  # iid -> history list

    for inst in instances:
        archive, best, history = solver.run(inst) 
        cost, viol = best.objectives

        print(f"\nInstance {inst.iid} (best-known: {inst.best_known})")
        print(f"  n={inst.n}, p={inst.p}, cap={inst.capacity}")
        print(f"  Best found: cost={cost:.4f}, violation={viol:.4f}")
        print(f"  Medians (0-based indices): {best.medians}")
        print(f"  Archive size: {len(archive)}")
        print("  Archive sample (cost, violation, fitness):")
        for ind in archive[: min(10, len(archive))]:
            print(f"    ({ind.objectives[0]:.4f}, {ind.objectives[1]:.4f}, F={ind.fitness:.4f})")

        histories[inst.iid] = history
        
    if instance_id is not None:
        return histories[instance_id]

    return histories

In [54]:
# Calles försök åt en SPEA algoritm 

@dataclass
class Individual:
    medians: List[int]
    objectives: Tuple[float, float]
    fitness: float = float("inf")           


def objective_distance(a: Individual, b: Individual) -> float:
    # Euclidean distance in objectiwe space
    (c1, v1) = a.objectives
    (c2, v2) = b.objectives
    return math.sqrt((c1 - c2) ** 2 + (v1 - v2) ** 2)


def determine_fitness_and_raw_strength(individuals: List[Individual], dist_matrix: np.ndarray, k: int): 
    fitness_scores = [0.0 for _ in range(len(individuals))]
    strengths = [0 for _ in range(len(individuals))]
    distances = [[(i, 0.0) for i in range(len(individuals))] for _ in range(len(individuals))]
    # first determine strength of all candidate solutions
    for i in range(len(individuals)): 
        for j in range(len(individuals)): 
            if i == j: continue 
            ind_a, ind_b = individuals[i], individuals[j]
            a_dist, a_violations = ind_a.objectives 
            b_dist, b_violations = ind_b.objectives 
            # First condition: candidate a is at least as good as candidate b in both measures 
            # Second condition: candidate a does better for at least one of the constraints than candidate b
            if (a_dist <= b_dist and a_violations <= b_violations) and (a_dist < b_dist or a_violations < b_violations): 
                strengths[i] += 1
            distances[i][j] = (j, objective_distance(ind_a, ind_b))

    for i in range(len(individuals)): 
        distances[i] = sorted(distances[i], key=lambda x: -x[1]) 
        knn = distances[i][k][1]
        density = 1/(knn + 2) 
        individuals[i].fitness = density

    # calculate fitness by calculating total strength of all individuals that dominate a candidate 
    for i in range(len(individuals)): 
        for j in range(len(individuals)): 
            if i == j: continue 
            ind_a, ind_b = individuals[i], individuals[j]
            a_dist, a_violations = ind_a.objectives 
            b_dist, b_violations = ind_b.objectives
            if (a_dist <= b_dist and a_violations <= b_violations) and (a_dist < b_dist or a_violations < b_violations): 
                individuals[j].fitness += strengths[i] 
    
    return individuals 

def create_new_generation(archive: List[Individual], population: List[Individual]):
    pass
        
    

In [68]:
def fill_none(hist, fallback=None):
    if fallback is None:
        non_none = [v for v in hist if v is not None]
        fallback = non_none[0] if non_none else 1e9
    return [fallback if v is None else v for v in hist]

In [56]:
#Results


In [69]:

data_path   = "p_median_capacitated.txt"
instance_id = 1

pop_size    = 200
generations = 400
pc          = 0.1
pm          = 0.25

demand_weighted = False
customer_order  = "desc_demand"
seed            = 1234

# SPEA2 config
metric = "euclidean"
rounding = "floor"
archive_size = 200

# --- run NSGA-II 
nsga_hist = run_nsga2(
    data_path,
    instance_id=instance_id,
    pop_size=pop_size,
    generations=generations,
    pc=pc,
    pm=pm,
    demand_weighted=demand_weighted,
    customer_order=customer_order,
    seed=seed
)

# --- run SPEA2 
spea_hist = run_spea2(
    data_path,
    instance_id,
    metric,
    rounding,
    pop_size=pop_size,
    archive_size=archive_size,
    generations=generations,
    pc=pc,
    pm=pm,
    demand_weighted=demand_weighted,
    customer_order=customer_order,
    seed=seed
)

nsga_y = fill_none(nsga_hist)
spea_y = fill_none(spea_hist)

L = min(len(nsga_y), len(spea_y))
nsga_y = nsga_y[:L]
spea_y = spea_y[:L]
x = list(range(L))

plt.figure()
plt.plot(x, nsga_y, label="NSGA-II best feasible cost")
plt.plot(x, spea_y, label="SPEA2 best feasible cost")
plt.xlabel("Generation")
plt.ylabel("Best feasible cost")
plt.title(f"Instance {instance_id}: NSGA-II vs SPEA2")
plt.legend()
plt.tight_layout()

out = f"nsga2_vs_spea2_instance{instance_id}.png"
plt.savefig(out, dpi=200)
plt.show()

print(f"\nSaved plot to: {out}")
print(f"Final NSGA-II best feasible cost: {nsga_y[-1]:.4f}")
print(f"Final SPEA2  best feasible cost: {spea_y[-1]:.4f}")

NameError: name 'dmd' is not defined